# EDA na Prática — Pokémon Stats

**Dataset:** Pokémon Stats — Kaggle  
**Fonte:** https://www.kaggle.com/datasets/ulrikthygepedersen/pokemon-stats

Nesta atividade será realizada uma Análise Exploratória de Dados (EDA) seguindo a mesma lógica vista em sala: estrutura dos dados, dados ausentes, distribuições, outliers, estatística descritiva, relações entre variáveis e síntese final.

O objetivo principal é entender como os atributos dos Pokémon se comportam e verificar relações entre **status base**, **tipo**, **geração** e o fato de o Pokémon ser **lendário ou não**.

## 1. Setup de bibliotecas e carregamento do dataset

O código abaixo tenta baixar diretamente o dataset público do Kaggle. Caso o download automático não funcione no Colab, ele pede o upload manual do arquivo CSV.

In [ ]:
!pip -q install kagglehub

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import entropy

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA = "lightgray"
COR_CONTEXTO = "#e9d4d0"
COR_DESTAQUE = "#2d223a"
COR_ALERTA = "tab:red"

In [ ]:
try:
    import kagglehub

    caminho = kagglehub.dataset_download("ulrikthygepedersen/pokemon-stats")
    arquivos = glob.glob(os.path.join(caminho, "**", "*.csv"), recursive=True)

    if len(arquivos) == 0:
        raise FileNotFoundError("Nenhum CSV encontrado no download do Kaggle.")

    arquivo_csv = arquivos[0]
    df = pd.read_csv(arquivo_csv)

    print("Arquivo carregado:", arquivo_csv)

except Exception as erro:
    print("Download automático não funcionou.")
    print("Faça o upload do pokemon.csv abaixo.")

    from google.colab import files
    uploaded = files.upload()

    arquivo_csv = next(iter(uploaded))
    df = pd.read_csv(arquivo_csv)

df.head()

## 2. Princípio Estrutural

Antes de analisar qualquer relação, precisamos entender a estrutura do conjunto: quantidade de linhas e colunas, tipos das variáveis, duplicadas e possíveis problemas de qualidade.

### Pergunta
O dataset possui apenas espécies únicas ou também inclui formas alternativas, como Mega Evoluções, formas regionais e Gigantamax?

### Discussão
O número da Pokédex pode aparecer mais de uma vez porque uma mesma espécie pode possuir diferentes formas. Por isso, repetir `number` não significa automaticamente que a linha é duplicada.

In [ ]:
print("Dimensão:", df.shape)
print()
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Linhas completamente duplicadas:", df.duplicated().sum())
print("Números da Pokédex repetidos:", df["number"].duplicated().sum())

In [ ]:
df[df["number"].duplicated(keep=False)].sort_values("number").head(20)

### Discussão
Linhas totalmente iguais podem ser consideradas duplicadas. Já os números da Pokédex repetidos serão mantidos, pois representam formas diferentes do mesmo Pokémon. Remover essas linhas faria perder informação real do dataset.

## 3. Dados ausentes e qualidade dos dados

Primeiro verificamos os valores ausentes. Em seguida, analisamos se eles realmente representam erro.

No domínio Pokémon, `type2` vazio geralmente significa que o Pokémon possui **apenas um tipo**, portanto não é um dado desconhecido no sentido tradicional.

In [ ]:
df.isna().sum()

In [ ]:
print("Valores únicos de type1:")
print(sorted(df["type1"].dropna().astype(str).unique()))

print("\nValores únicos de type2:")
print(sorted(df["type2"].dropna().astype(str).unique()))

### Pergunta
Os valores ausentes em `type2` devem ser removidos ou preenchidos por média/mediana?

### Resposta
Não. `type2` é uma variável qualitativa e a ausência possui significado no próprio domínio: o Pokémon tem somente um tipo. Por isso, será criada a categoria **"Sem segundo tipo"** em vez de excluir essas observações.

In [ ]:
df_antes = df.copy()

# Padroniza strings vazias como ausentes
df["type2"] = df["type2"].replace("", np.nan)

# Correções de erros conhecidos de digitação/categoria no próprio arquivo
df["type1"] = df["type1"].replace({"Graass": "Grass"})

mascara_blastoise = df["type1"].eq("Blastoise")
df.loc[mascara_blastoise, "type1"] = "Water"
df.loc[mascara_blastoise, "type2"] = np.nan

# Correção apenas do nome digitado incorretamente, quando existir
df["name"] = df["name"].replace({"Gigantamax Blasoise": "Gigantamax Blastoise"})

# Ausência estrutural: Pokémon de um único tipo
df["type2"] = df["type2"].fillna("Sem segundo tipo")

print("Ausentes depois do tratamento:")
print(df.isna().sum())

print("\nTipos primários após correção:")
print(sorted(df["type1"].unique()))

### Discussão
Aqui foram usados dois tratamentos diferentes:

- ausência em `type2`: tratada como uma característica legítima, criando a categoria `"Sem segundo tipo"`;
- erros como `"Graass"` e `"Blastoise"` aparecendo como tipo primário: tratados como erro de digitação/classificação.

Essa diferença é importante porque nem todo valor ausente ou estranho deve ser tratado da mesma forma.

## 4. Princípio Distribucional

Vamos observar as variáveis numéricas principais. A coluna `total` representa a soma dos seis status base (`hp`, `attack`, `defense`, `sp_attack`, `sp_defense` e `speed`).

In [ ]:
VARS_QUANT = [
    "total", "hp", "attack", "defense",
    "sp_attack", "sp_defense", "speed"
]

VARS_QUALI = ["type1", "type2", "generation", "legendary"]

desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc["kurtosis"] = df[VARS_QUANT].kurtosis()

desc[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]].rename(
    columns={"50%": "median"}
).round(2)

### Perguntas para observar
- Quais status apresentam maior dispersão?
- Média e mediana ficam próximas?
- Existem variáveis com cauda longa?
- Os valores extremos parecem erros ou Pokémon realmente fora do padrão?

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for i, col in enumerate(VARS_QUANT):
    sns.histplot(data=df, x=col, kde=True, color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(col)

for ax in axes[len(VARS_QUANT):]:
    ax.axis("off")

fig.suptitle("Distribuição dos status dos Pokémon", fontsize=15)
plt.tight_layout()
plt.show()

### Normalidade — Shapiro-Wilk

Vamos testar a normalidade das variáveis quantitativas. Como temos mais de mil observações, o teste pode detectar até desvios pequenos, então a interpretação será feita junto dos histogramas.

In [ ]:
resultados_shapiro = []

for col in VARS_QUANT:
    stat, p = stats.shapiro(df[col].dropna())

    resultados_shapiro.append({
        "variavel": col,
        "estatistica": round(stat, 4),
        "p_valor": p,
        "rejeita_normalidade_5%": p < 0.05
    })

pd.DataFrame(resultados_shapiro).set_index("variavel")

### Discussão
Se `p < 0,05`, temos evidência contra a hipótese de normalidade. Para comparações posteriores, vamos preferir testes não paramétricos quando a normalidade não for adequada.

### ECDF — Total de status por classe lendária

In [ ]:
df["legendary_label"] = df["legendary"].map(
    {True: "Lendário", False: "Não lendário", "TRUE": "Lendário", "FALSE": "Não lendário"}
)

# Caso legendary tenha sido lido como texto com outra capitalização
df["legendary_label"] = df["legendary_label"].fillna(
    df["legendary"].astype(str).str.upper().map(
        {"TRUE": "Lendário", "FALSE": "Não lendário"}
    )
)

plt.figure(figsize=(8, 5))
sns.ecdfplot(
    data=df,
    x="total",
    hue="legendary_label",
    palette={"Não lendário": COR_CONTEXTO, "Lendário": COR_DESTAQUE}
)
plt.title("ECDF — Total de status por classe")
plt.xlabel("Total de status")
plt.ylabel("Proporção acumulada")
plt.tight_layout()
plt.show()

### Média e mediana do total

In [ ]:
media_total = df["total"].mean()
mediana_total = df["total"].median()

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="total", kde=True, color=COR_NEUTRA)
plt.axvline(media_total, color=COR_ALERTA, linestyle="--", linewidth=2,
            label=f"Média = {media_total:.1f}")
plt.axvline(mediana_total, color="black", linewidth=2,
            label=f"Mediana = {mediana_total:.1f}")
plt.title("Total de status — média e mediana")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Dispersão e outliers

A regra do IQR considera outlier qualquer valor menor que `Q1 - 1,5 × IQR` ou maior que `Q3 + 1,5 × IQR`.

In [ ]:
def limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    return limite_inferior, limite_superior

resumo_outliers = {}

for col in VARS_QUANT:
    lim_inf, lim_sup = limites_iqr(df[col])

    n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()

    resumo_outliers[col] = {
        "limite_inferior": round(lim_inf, 2),
        "limite_superior": round(lim_sup, 2),
        "n_outliers": int(n_outliers)
    }

pd.DataFrame(resumo_outliers).T

In [ ]:
fig, axes = plt.subplots(1, len(VARS_QUANT), figsize=(21, 5))

for i, col in enumerate(VARS_QUANT):
    sns.boxplot(
        data=df,
        y=col,
        color=COR_NEUTRA,
        flierprops={
            "markerfacecolor": COR_ALERTA,
            "markeredgecolor": COR_ALERTA,
            "marker": "o"
        },
        ax=axes[i]
    )
    axes[i].set_title(col)

fig.suptitle("Outliers pela regra do IQR", fontsize=15)
plt.tight_layout()
plt.show()

### Investigando os maiores valores

Antes de apagar qualquer outlier, verificamos quais Pokémon estão gerando esses valores.

In [ ]:
for col in ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]:
    print(f"\n--- Maiores valores de {col} ---")
    display(
        df[["name", "type1", "legendary_label", col]]
        .sort_values(col, ascending=False)
        .head(5)
    )

### Decisão sobre os outliers
Os valores extremos serão **mantidos**.

Eles são estatisticamente raros, mas fazem sentido dentro do universo Pokémon. Alguns Pokémon são propositalmente especializados em HP, defesa, ataque ou velocidade. Portanto, remover esses valores significaria apagar características reais do domínio.

Classificação: **outliers estatísticos legítimos**.

## 6. Estatística descritiva depois do tratamento

O tratamento realizado foi principalmente categórico: correção de nomes/tipos e identificação correta de Pokémon com apenas um tipo. Os valores numéricos não foram alterados, então esperamos pouca ou nenhuma mudança nas estatísticas quantitativas.

In [ ]:
desc_depois = df[VARS_QUANT].describe().T
desc_depois["skewness"] = df[VARS_QUANT].skew()
desc_depois["kurtosis"] = df[VARS_QUANT].kurtosis()

desc_depois[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]].rename(
    columns={"50%": "median"}
).round(2)

In [ ]:
comparacao = pd.DataFrame({
    "media_antes": df_antes[VARS_QUANT].mean(),
    "media_depois": df[VARS_QUANT].mean(),
    "mediana_antes": df_antes[VARS_QUANT].median(),
    "mediana_depois": df[VARS_QUANT].median()
})

comparacao.round(3)

### Discussão
Como os erros encontrados estavam principalmente nas variáveis categóricas e os outliers numéricos foram mantidos, as médias e medianas dos status praticamente não mudam. Isso confirma que o tratamento foi conservador.

## 7. Variáveis qualitativas

Agora observamos frequências dos tipos, gerações e da classe lendária.

In [ ]:
for col in VARS_QUALI:
    print(f"\n--- {col} ---")
    freq_abs = df[col].value_counts(dropna=False)
    freq_rel = df[col].value_counts(normalize=True, dropna=False).round(3)

    display(pd.concat([
        freq_abs.rename("absoluta"),
        freq_rel.rename("relativa")
    ], axis=1))

In [ ]:
ordem_tipo = df["type1"].value_counts().index

plt.figure(figsize=(12, 6))
sns.countplot(data=df, x="type1", order=ordem_tipo, color=COR_NEUTRA)
plt.title("Quantidade de Pokémon por tipo primário")
plt.xlabel("Tipo primário")
plt.ylabel("Quantidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="generation", color=COR_NEUTRA)
plt.title("Quantidade de registros por geração")
plt.xlabel("Geração")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()

### Desbalanceamento da variável `legendary`

In [ ]:
contagens = df["legendary_label"].value_counts()
razao = contagens.max() / contagens.min()

props = contagens / contagens.sum()
entropia_normalizada = entropy(props, base=2) / np.log2(len(contagens))

print(contagens)
print()
print("Razão de desbalanceamento:", round(razao, 2))
print("Entropia normalizada:", round(entropia_normalizada, 3))

### Discussão
É esperado que a categoria **não lendário** seja muito mais frequente. Isso precisa ser lembrado ao interpretar gráficos e testes, porque os dois grupos possuem tamanhos bastante diferentes.

# 8. Princípio Relacional

## 8.1 Quantitativa × quantitativa — correlação

Primeiro mostramos a correlação entre as variáveis. `total` deve ter correlação positiva com vários atributos porque ele é calculado a partir da soma deles. Por isso, essa relação precisa ser interpretada com cuidado.

In [ ]:
corr_pearson = df[VARS_QUANT].corr(method="pearson")
corr_spearman = df[VARS_QUANT].corr(method="spearman")
corr_kendall = df[VARS_QUANT].corr(method="kendall")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.heatmap(corr_pearson, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson")

sns.heatmap(corr_spearman, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman")

sns.heatmap(corr_kendall, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, ax=axes[2])
axes[2].set_title("Kendall")

plt.tight_layout()
plt.show()

### Correlação apenas entre os seis status base

Aqui retiramos `total` para evitar que uma variável que já é a soma das outras domine a interpretação.

In [ ]:
BASE_STATS = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]

plt.figure(figsize=(8, 6))
sns.heatmap(
    df[BASE_STATS].corr(method="spearman"),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Correlação de Spearman — status base")
plt.tight_layout()
plt.show()

### Pergunta
Existe algum par de status com associação forte o suficiente para sugerir que Pokémon fortes em uma característica também tendem a ser fortes em outra?

### Discussão
A matriz permite identificar essas relações sem confundir a análise com `total`, que é uma variável derivada. Correlação não significa que um atributo causa o outro.

## 8.2 Dispersão entre dois status

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="attack",
    y="sp_attack",
    hue="legendary_label",
    palette={"Não lendário": COR_CONTEXTO, "Lendário": COR_DESTAQUE},
    alpha=0.75
)
plt.title("Ataque físico × Ataque especial")
plt.xlabel("Attack")
plt.ylabel("Sp. Attack")
plt.tight_layout()
plt.show()

### Discussão
O gráfico ajuda a observar especializações. Alguns Pokémon possuem ataque físico alto e ataque especial baixo, enquanto outros fazem o contrário. Os lendários também podem ocupar regiões de valores mais elevados, mas isso deve ser confirmado com um teste formal.

## 8.3 Quantitativa × qualitativa — Total por classe lendária

Vamos comparar `total` entre Pokémon lendários e não lendários.

Usaremos:
- **Welch T-Test** como contraste paramétrico;
- **Mann-Whitney U** como teste não paramétrico principal quando a normalidade não for adequada.

In [ ]:
grupo_nao = df.loc[df["legendary_label"] == "Não lendário", "total"]
grupo_sim = df.loc[df["legendary_label"] == "Lendário", "total"]

stat_t, p_t = stats.ttest_ind(grupo_nao, grupo_sim, equal_var=False)
stat_u, p_u = stats.mannwhitneyu(grupo_nao, grupo_sim, alternative="two-sided")

print("Welch T-Test")
print("estatística =", round(stat_t, 4))
print("p-valor =", p_t)

print("\nMann-Whitney U")
print("estatística =", round(stat_u, 4))
print("p-valor =", p_u)

print("\nMediana não lendário:", grupo_nao.median())
print("Mediana lendário:", grupo_sim.median())

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=df,
    x="legendary_label",
    y="total",
    hue="legendary_label",
    palette={"Não lendário": COR_CONTEXTO, "Lendário": COR_DESTAQUE},
    legend=False
)
plt.title("Total de status — lendários × não lendários")
plt.xlabel("")
plt.ylabel("Total")
plt.tight_layout()
plt.show()

### Discussão
Se o Mann-Whitney apresentar `p < 0,05`, existe evidência estatística de diferença na distribuição do `total` entre os dois grupos.

Além do p-valor, devemos observar as medianas e o boxplot, pois significância estatística sozinha não informa o tamanho visual da diferença.

## 8.4 Exemplo com 3 ou mais grupos — Total por geração

Vamos verificar se a distribuição do `total` é igual entre as gerações.

- ANOVA: teste paramétrico;
- Kruskal-Wallis: alternativa não paramétrica.

In [ ]:
grupos_geracao = [
    grupo["total"].values
    for _, grupo in df.groupby("generation")
]

stat_anova, p_anova = stats.f_oneway(*grupos_geracao)
stat_kw, p_kw = stats.kruskal(*grupos_geracao)

print("ANOVA")
print("estatística =", round(stat_anova, 4))
print("p-valor =", p_anova)

print("\nKruskal-Wallis")
print("estatística =", round(stat_kw, 4))
print("p-valor =", p_kw)

In [ ]:
media_por_geracao = df.groupby("generation")["total"].mean().sort_index()

plt.figure(figsize=(9, 5))
media_por_geracao.plot(kind="bar")
plt.title("Média do total de status por geração")
plt.xlabel("Geração")
plt.ylabel("Média do total")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(media_por_geracao.round(2).to_frame("media_total"))

### Discussão
Se o Kruskal-Wallis apresentar `p < 0,05`, existe evidência de que pelo menos uma geração possui distribuição de `total` diferente das demais. O teste não informa sozinho quais pares de gerações são diferentes.

## 8.5 Qualitativa × qualitativa — Tipo primário × Lendário

In [ ]:
contingencia_tipo = pd.crosstab(df["type1"], df["legendary_label"])
chi2, p_chi2, gl, esperado = stats.chi2_contingency(contingencia_tipo)

print("Qui-quadrado")
print("estatística =", round(chi2, 4))
print("graus de liberdade =", gl)
print("p-valor =", p_chi2)

plt.figure(figsize=(7, 10))
sns.heatmap(contingencia_tipo, annot=True, fmt="d", cmap="magma")
plt.title(f"Tipo primário × Lendário (p={p_chi2:.4g})")
plt.xlabel("Classe")
plt.ylabel("Tipo primário")
plt.tight_layout()
plt.show()

### Discussão
Se `p < 0,05`, tipo primário e classe lendária apresentam associação estatística. Isso não significa que o tipo "cause" um Pokémon a ser lendário; apenas indica que a distribuição de lendários não é igual entre todos os tipos.

## 9. Representatividade do conjunto

A seção de fairness usada no exemplo médico não pode ser aplicada da mesma forma aqui, pois Pokémon não são pessoas e não existem atributos humanos protegidos no dataset.

Mesmo assim, podemos avaliar **representatividade**: algumas gerações e tipos podem aparecer muito mais que outros, o que pode influenciar análises ou modelos futuros.

In [ ]:
tabela_rep = pd.DataFrame({
    "quantidade": df["generation"].value_counts().sort_index(),
    "proporcao": df["generation"].value_counts(normalize=True).sort_index().round(3)
})

tabela_rep

In [ ]:
prop_lendario_geracao = pd.crosstab(
    df["generation"],
    df["legendary_label"],
    normalize="index"
).round(3)

prop_lendario_geracao

### Discussão
Diferenças de frequência entre gerações ou tipos precisam ser lembradas antes de criar qualquer modelo. Uma classe com muito mais exemplos pode influenciar métricas e padrões aprendidos.

## 10. Outliers bivariados

Um ponto pode não parecer extremo quando cada variável é analisada sozinha, mas pode formar uma combinação incomum quando duas características são observadas juntas.

In [ ]:
sns.jointplot(
    data=df,
    x="attack",
    y="defense",
    hue="legendary_label",
    palette={"Não lendário": COR_CONTEXTO, "Lendário": COR_DESTAQUE},
    alpha=0.7,
    height=7
)
plt.suptitle("Attack × Defense — combinações incomuns", y=1.02)
plt.show()

## 11. Análise multivariada

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    df[BASE_STATS].corr(method="spearman"),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.title("Matriz de correlação multivariada — Spearman")
plt.tight_layout()
plt.show()

In [ ]:
amostra_pairplot = df.sample(min(500, len(df)), random_state=42)

sns.pairplot(
    amostra_pairplot,
    vars=BASE_STATS,
    hue="legendary_label",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 25}
)
plt.show()

### Como interpretar
Como referência:

- `|r| < 0,3`: correlação fraca;
- `0,3 ≤ |r| < 0,7`: correlação moderada;
- `|r| ≥ 0,7`: correlação forte.

A interpretação precisa considerar que diferentes Pokémon possuem funções e especializações diferentes, então nem todos os status precisam crescer juntos.

# 12. Síntese — principais conclusões

A célula abaixo calcula automaticamente alguns resultados importantes para que as conclusões fiquem baseadas no próprio dataset executado.

In [ ]:
medianas = df.groupby("legendary_label")["total"].median()

media_geracao = df.groupby("generation")["total"].mean()
geracao_maior_media = media_geracao.idxmax()
valor_maior_media = media_geracao.max()

corr_base = df[BASE_STATS].corr(method="spearman")
triangulo = corr_base.where(np.triu(np.ones(corr_base.shape), k=1).astype(bool))
par_maior = triangulo.abs().stack().idxmax()
corr_maior = corr_base.loc[par_maior[0], par_maior[1]]

tipo_mais_comum = df["type1"].value_counts().idxmax()
qtd_tipo_mais_comum = df["type1"].value_counts().max()

print("1. O dataset possui", len(df), "registros após o tratamento, mantendo formas alternativas como observações válidas.")
print("2. Valores vazios em type2 representam principalmente Pokémon de um único tipo e foram tratados como 'Sem segundo tipo'.")
print("3. Mediana do total — Não lendário:", medianas.get("Não lendário"),
      "| Lendário:", medianas.get("Lendário"))
print("4. Geração com maior média de total:", geracao_maior_media,
      "| média =", round(valor_maior_media, 2))
print("5. Maior correlação de Spearman entre status base:",
      par_maior[0], "×", par_maior[1], "=", round(corr_maior, 3))
print("6. Tipo primário mais frequente:", tipo_mais_comum,
      "| registros =", qtd_tipo_mais_comum)

### Conclusões em 3 a 5 pontos

1. O conjunto contém formas alternativas de Pokémon, portanto números da Pokédex repetidos não devem ser removidos automaticamente como duplicatas.
2. A principal ausência ocorre em `type2`, mas ela possui significado no domínio: muitos Pokémon têm apenas um tipo. Por isso, foi usada uma categoria específica em vez de excluir linhas.
3. Os valores extremos dos status foram mantidos porque representam Pokémon realmente especializados e não erros evidentes de medição.
4. O grupo de Pokémon lendários apresenta um perfil de `total` diferente dos não lendários; a comparação formal foi realizada com Mann-Whitney, apoiada pelo boxplot e pelas medianas.
5. `total` possui relação forte com os status individuais por construção, já que é a soma deles. Para analisar relações mais informativas, também foi usada uma matriz apenas com os seis status base.

# 13. Declaração de Uso de IA

**Ferramenta utilizada:** ChatGPT (OpenAI).

**Para o que foi aplicada:** apoio na adaptação do roteiro de EDA apresentado em sala para o dataset Pokémon Stats, organização das etapas, elaboração e revisão de código Python, escolha de gráficos e testes estatísticos e apoio na redação das discussões.

**O que foi aceito/modificado/rejeitado:** foi aceita a estrutura geral de análise e os códigos que foram conferidos no contexto do dataset. Foram modificadas decisões de tratamento para respeitar o significado das variáveis de Pokémon, principalmente `type2`. Foi rejeitada a ideia de remover automaticamente todos os outliers, porque os valores extremos são plausíveis e representam características reais de determinados Pokémon.